# Using Causal Reasoning to Test Scientific Software

## Michael Foster and Sylvia Whittle

m.foster@sheffield.ac.uk &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; sylvia.whittle@sheffield.ac.uk

# Motivating Example: Covasim
- Stochastic agent-based simulator for performing COVID-19 analyses
- Used to inform research studies and policy decisions in multiple countries including the US, UK, and Australia
- Important to make sure the model is working properly

In [ ]:
import covasim as cv

sim = cv.Sim(location="UK", beta=0.016, pop_type="hybrid", verbose=0)
sim.run()
sim.summarize()

# A basic regression test

In [ ]:
EXPECTED_VALUE = 10800


def cumulative_infections(sim: cv.Sim):
    return int(sim.results["cum_infections"][-1])


def test_run_uk():
    sim = cv.Sim(location="UK", beta=0.016, pop_type="hybrid", verbose=0)
    sim.run()
    final_infections = cumulative_infections(sim)
    assert final_infections == EXPECTED_VALUE, f"{final_infections} != {EXPECTED_VALUE}"
    print(f"{final_infections} == {EXPECTED_VALUE}")


test_run_uk()

# Is the model working correctly?
If it is, we should see 10,800 every time we run the simulator...

In [ ]:
msim = cv.MultiSim(cv.Sim(location="UK", beta=0.016, pop_type="hybrid", verbose=0))
msim.run(n_runs=5)
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {cumulative_infections(sim)} total infections")

**So which is correct?**

# Naive solution: Adding a tolerance

Instead of asserting `output == EXPECTED_VALUE`, assert output is _approximately_ the expected value.

In [ ]:
tolerance = EXPECTED_VALUE * 0.1
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {cumulative_infections(sim)} total infections")
    assert (
        (EXPECTED_VALUE - tolerance)
        < cumulative_infections(sim)
        < (EXPECTED_VALUE + tolerance)
    )

Choosing too small a tolerance will lead to failing tests

In [ ]:
tolerance = EXPECTED_VALUE * 0.4
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {sim.results['cum_infections'][-1]:,.0f} total infections")
    assert (
        (EXPECTED_VALUE - tolerance)
        < sim.results["cum_infections"][-1]
        < (EXPECTED_VALUE + tolerance)
    )

To get the test to pass, we need a tollerance of 40%!
- Is this _expected_?
- Is this test _meaningful_?

# What happens with a more infectious variant?

What do we expect to happen?

In [ ]:
msim = cv.MultiSim(cv.Sim(location="UK", beta=0.017, verbose=0))
msim.run(n_runs=5)
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {cumulative_infections(sim)} total infections")

**Are these results what we'd expect?**

# Metamorphic testing

Instead of asserting that `f(x) = y`, assert that _changing_ `x` leads to a _corresponding change in_ `y`.

`beta_1 > beta_2 ==> cum_infections_1 > cum_infections_2`

In [ ]:
msim_1 = cv.MultiSim(cv.Sim(location="UK", beta=0.016, verbose=0, pop_type="hybrid"))
msim_2 = cv.MultiSim(cv.Sim(location="UK", beta=0.017, verbose=0, pop_type="hybrid"))

msim_1.run(n_runs=5)
msim_2.run(n_runs=5)
for sim_1, sim_2 in zip(msim_1.sims, msim_2.sims):
    cum_infections_1 = cumulative_infections(sim_1)
    cum_infections_2 = cumulative_infections(sim_2)
    assert (
        cum_infections_2 > cum_infections_1
    ), f"{cum_infections_2} was less than {cum_infections_1}"

**Does this mean that there's a problem?**

Not necessarily - remember covasim is inherently stochastic

# Statistical metamorphic testing

Instead of testing relationships between exact values, test relationships between _populatiuons_ of runs.

In [ ]:
import numpy as np


def test_population_means(n_runs: int):
    msim_1 = cv.MultiSim(
        cv.Sim(location="UK", beta=0.016, verbose=0, pop_type="hybrid")
    )
    msim_2 = cv.MultiSim(
        cv.Sim(location="UK", beta=0.017, verbose=0, pop_type="hybrid")
    )

    msim_1.run(n_runs=n_runs)
    msim_2.run(n_runs=n_runs)

    cum_infections_1 = [cumulative_infections(sim) for sim in msim_1.sims]
    cum_infections_2 = [cumulative_infections(sim) for sim in msim_2.sims]

    assert np.mean(cum_infections_2) > np.mean(
        cum_infections_1
    ), f"{np.mean(cum_infections_2)} < {np.mean(cum_infections_1)}"
    print(f"{np.mean(cum_infections_2)} > {np.mean(cum_infections_1)}")


test_population_means(n_runs=5)

# Is this feasible?

Running with five repeats isn't very statistically significant. Let's try it again with 30 repeats for each.

In [ ]:
from time import time

start = time()
test_population_means(30)
end = time()
print(f"That took {end - start} seconds!")

Now let's test every pair of the 202 supported countries - that's 20,301 combinations, so 182,709 seconds: >2 days!

Covasim has more than 60 parameters, many of which are complex objects with their own sub-parameters.

Collecting the data to test all these relationships could take months!

# Causal testing

Causal testing aleviates this problem by allowing us to re-use data.

The process of data collection (running the model) is completely separate from performing the tests (performing the validation).

The [Causal Testing Framework](https://github.com/CITCOM-project/CausalTestingFramework) provides a suite of tools to facilitate causal testing.

![Causal testing framework workflow](https://github.com/CITCOM-project/CausalTestingFramework/raw/main/images/schematic.png)

# Specify the expected causal relationships

A _directed acyclic graph_ (DAG) shows the expected relationships between model parameters and outputs.7

An edge `X -> Y` represents that `X` causes `Y`, i.e. that the value of `Y` somehow depends on the value of `X`.

In [ ]:
%cat simple_dag.dot

from causal_testing.specification.causal_dag import CausalDAG
import networkx as nx
from IPython.display import HTML, display


def render_dag(dag: CausalDAG, size=None):
   rendered = nx.nx_agraph.to_agraph(dag)
   if size is not None:
       rendered.graph_attr.update(size=size)
   rendered.layout(prog="dot")
   
   # Get raw SVG string
   svg_data = rendered.draw(format='svg').decode('utf-8')
   
   # Inject CSS to make edge lines easier to hover over
   hover_css = """
   <style>
       .edge path {
           stroke-width: 3px !important;  /* Make line slightly thicker */
           cursor: pointer;
       }
       .edge:hover path {
           stroke-width: 5px !important;
       }
   </style>
   """
   
   display(HTML(hover_css + svg_data)) 


simple_dag = CausalDAG("simple_dag.dot")

render_dag(simple_dag)

# Covasim is actually a little more complex

The location does not determine the cumulative infections directly.

Instead, it determines the average age of the population and the number of contacts each agent has at home, school, work, and in the community.

The user has no direct control over these parameters from outside the model. They can only change the location.

In [ ]:
%cat complex_dag.dot

complex_dag = CausalDAG("complex_dag.dot")
render_dag(complex_dag)

# Collecting data

Run the model a bunch of times with random values for each parameter

In [ ]:
import pandas as pd
import random

random.seed(0)

RUNS = 10
locations = list(cv.data.country_age_data.data)
betas = np.linspace(0.010, 0.020, RUNS)  # Sweep beta from 0.01 to 0.02 with 5 values
msim = cv.MultiSim(
    [
        cv.Sim(
            beta=beta, location=random.choice(locations), pop_type="hybrid", verbose=0
        )
        for beta in betas
    ]
)
msim.run(keep_people=True)
data = []
for sim in msim.sims:
    datum = {
        "location": sim.pars["location"],
        "beta": sim.pars["beta"],
        "average_age": sim.people["age"].mean(),
        "contacts_home": sim.pars["contacts"]["h"],
        "contacts_school": sim.pars["contacts"]["s"],
        "contacts_work": sim.pars["contacts"]["w"],
        "contacts_community": sim.pars["contacts"]["c"],
        "cum_infections": cumulative_infections(sim),
    }
    data.append(datum)
data = pd.DataFrame(data)

# Generating causal tests

In [ ]:
complex_dag.datatypes = data.dtypes
causal_tests = complex_dag.generate_causal_tests()
causal_tests[0].to_dict()

# Running test cases

In [ ]:
from causal_testing.causal_testing_framework import CausalTestingFramework
import warnings

# hide warnings that occur as a result of causal test adequacy calculation
warnings.filterwarnings("ignore")

ctf = CausalTestingFramework(dag=complex_dag, df=data, test_cases=causal_tests)
ctf.run_tests(silent=True, adequacy=True)

# Visualising the results

In [ ]:
from causal_testing.visualisation.causal_test_result_visualiser import results_dag

results = results_dag(dag=ctf.dag, test_cases=ctf.test_cases)

render_dag(results, size="10,10")

[test] = [t for t in ctf.test_cases if t.treatment_variable == "location" and t.outcome_variable == "contacts_home"]
print(test.to_dict())

# Interpreting the results

Just because a test has failed doesn't mean there's a problem with the model! There could be a problem with our causal DAG or we may not have enough data.

# Causal Test Adequacy

- When a test fails, there may not be a problem with the software or with our causal DAG.
- It may just be that we haven't collected enough runs to calculate a reliable causal estimate.
- Causal Test Adequacy tells us how trustworthy our causal test outcomes are.

In [ ]:
from causal_testing.testing.causal_test_case import CausalTestCase
import holoviews as hv
from matplotlib.colors import TwoSlopeNorm

hv.extension('bokeh')

def data_adequacy_heatmap(test_cases: list[CausalTestCase]):
    adequacy = pd.json_normalize(map(lambda t: t.to_dict(), ctf.test_cases))

    for col in ["effect_estimate", "ci_low", "ci_high", "adequacy.kurtosis"]:
        columns = [c for c in adequacy.columns if c.startswith(f"result.{col}.")]
        adequacy[f"result.{col}"] = adequacy[columns].bfill(axis=1).iloc[:, 0]
        adequacy = adequacy.drop(columns=columns)

    # Get data bounds
    vmin = adequacy["result.adequacy.kurtosis"].min()
    vmax = adequacy["result.adequacy.kurtosis"].max()

    # Calculate zero position (0.0 to 1.0)
    zero_ratio = (0 - vmin) / (vmax - vmin)

    # 1000 samples, proportionally assigned negative and positive
    N = 1000
    n_neg = max(1, int(N * zero_ratio))
    n_pos = N - n_neg

    # Generate the colour samples from the negative and positive colourmaps
    neg_colors = hv.plotting.util.process_cmap('blues_r', provider='bokeh', ncolors=n_neg)
    pos_colors = hv.plotting.util.process_cmap('YlOrRd', provider='bokeh', ncolors=n_pos)

    # Join the hex lists together
    asymmetric_cmap = neg_colors + pos_colors

    # Render
    return hv.HeatMap(adequacy, kdims=["estimator.treatment_variable", "estimator.outcome_variable"], vdims=["result.adequacy.kurtosis"]).opts(
        cmap=asymmetric_cmap,
        clim=(vmin, vmax),
        # Grey out invalid tests
        clipping_colors={'NaN': 'grey'},
        colorbar=True,
        xrotation=90,
        width=600,
        height=500,
        tools=["hover"],
    )

data_adequacy_heatmap(ctf.test_cases)

# Evaluating causal DAGs

We can see how well a given causal DAG fits the data we have, and estimate the confidence that it's accurate.

In [ ]:
def dag_adequacy_heatmap(test_cases):
    adequacy = pd.json_normalize(map(lambda t: t.to_dict(), ctf.test_cases))

    for col in ["effect_estimate", "ci_low", "ci_high", "adequacy.kurtosis"]:
        columns = [c for c in adequacy.columns if c.startswith(f"result.{col}.")]
        adequacy[f"result.{col}"] = adequacy[columns].bfill(axis=1).iloc[:, 0]
        adequacy = adequacy.drop(columns=columns)

    return hv.HeatMap(adequacy, kdims=["estimator.treatment_variable", "estimator.outcome_variable"], vdims=["result.adequacy.passing"]).opts(
        cmap="RdYlGn",
        clim=(0, adequacy["result.adequacy.bootstrap_size"].max()),
        # Grey out invalid tests
        clipping_colors={'NaN': 'grey'},
        colorbar=True,
        xrotation=90,
        width=600,
        height=500,
        tools=["hover"],
    )

dag_adequacy_heatmap(ctf.test_cases)

# Inferring causal DAGs

We can also infer causal DAGs from the data.

In [ ]:
from causal_testing.discovery.hill_climber_discovery import HillClimberDiscovery

hill_climber = HillClimberDiscovery(df=data)
discovered_dag = hill_climber.discover()
render_dag(discovered_dag)

This DAG doesn't make much sense:

- `location` and `beta` are inputs, so are independent of the other variables
- `cum_infections` is an output, so cannot cause anything else

Let's add this knowledge to the discovery

In [ ]:
hill_climber = HillClimberDiscovery(df=data, exclude_edges=[(".*", "beta"), (".*", "location"), ("cum_infections", ".*")])
discovered_dag = hill_climber.discover()
render_dag(discovered_dag)

In [ ]:
discovered_ctf = CausalTestingFramework(dag=discovered_dag, df=data, test_cases=discovered_dag.generate_causal_tests())
discovered_ctf.run_tests(silent=True, adequacy=True)

In [ ]:
data_adequacy_heatmap(discovered_ctf.test_cases)

In [ ]:
dag_adequacy_heatmap(discovered_ctf.test_cases)

# Conclusion

- Scientific software is inherently hard to test
- Instead of asserting that a particular input configuration results in a particular output configuration, we can test the _relationships_ between inputs and outputs.
- Collecting repeated test runs for every parameter configuration we want to test can be infeasible.
- Causal testing allows us to maximise what we can do with the test runs we are able to collect.
- The Causal Testing Framework automates much of this process.